In [1]:
from google.colab import drive
drive.mount('/content/drive')
# mounting the google drive

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import tensorflow as tf # IMPORT CORE TENSOR FLOW LIB WHICH HANDLES THE BACKEND MATHS AND TENSOR OPERATIONS
from tensorflow.keras import layers, models # THIS IMPORT BASICALLY BUILD INDIVIDUAL NETWORK LAYERS
import os

# THIS IMPORT CONV2D , MAX POOLING  FOR STANDARD SPATIAL OPERATIONS FOR IMAGE PROCESSING
# CONCATENATE MULTIPLY : FOR COMBINING DIFFERENT STREAM OF dATA
# dENSE : FULLY CONNECTED LAYERS
# gLOBALaVERAGE POLLING , rEHSAPE : FOR CHANGING the shape and the dimensionality of the data tensor
from tensorflow.keras.layers import Conv2D, MaxPooling2D, concatenate, Dense, GlobalAveragePooling2D, Reshape, Dense, multiply
from tensorflow.keras import Model
from tensorflow.keras import backend as K # gives access to the



In [3]:

tomato_dir="/content/drive/MyDrive/plant_disease_dataset/data_/Tomato_"
print(f"Checking contents of: {tomato_dir}")

# List contents of the base dataset path (e.g., train, validation, test folders)
if os.path.exists(tomato_dir):
    print(f"Contents found at {tomato_dir}:")
    for item in os.listdir(tomato_dir):
        item_path = os.path.join(tomato_dir, item)
        print(f"- {item} {'(Directory)' if os.path.isdir(item_path) else '(File)'}")
else:
    print(f"Base dataset path does not exist: {tomato_dir}")


Checking contents of: /content/drive/MyDrive/plant_disease_dataset/data_/Tomato_
Contents found at /content/drive/MyDrive/plant_disease_dataset/data_/Tomato_:
- train (Directory)
- valid (Directory)


In [4]:
train_dir = os.path.join(tomato_dir, 'train') # Assuming 'train' is the directory with class folders


print(f"Folders in '{train_dir}':")

# List all entries in the train_dir
for item in os.listdir(train_dir):
    item_path = os.path.join(train_dir, item)
    # Check if the item is a directory (a class folder)
    if os.path.isdir(item_path):
        print(f"- {item}")

Folders in '/content/drive/MyDrive/plant_disease_dataset/data_/Tomato_/train':
- Septoria_leaf_spot
- powdery_mildew
- Tomato_mosaic_virus
- Bacterial_spot
- Early_blight
- Late_blight
- Leaf_Mold
- Spider_mites Two-spotted_spider_mite
- Tomato_Yellow_Leaf_Curl_Virus
- healthy
- Target_Spot


In [5]:



if os.path.exists(train_dir):
    print(f"\nCounting files in each class directory within: {train_dir}")
    class_counts = {}
    for class_name in os.listdir(train_dir):
        class_path = os.path.join(train_dir, class_name)
        if os.path.isdir(class_path):
            # Count only image files (e.g., .jpg, .jpeg, .png)
            num_files = len([name for name in os.listdir(class_path) if name.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))])
            class_counts[class_name] = num_files

    # Sort for consistent output
    sorted_class_counts = sorted(class_counts.items())

    for class_name, count in sorted_class_counts:
        print(f"{class_name}: {count} files")
else:
    print(f"The directory '{train_dir}' does not exist. Please check the dataset structure.")


Counting files in each class directory within: /content/drive/MyDrive/plant_disease_dataset/data_/Tomato_/train
Bacterial_spot: 2826 files
Early_blight: 2455 files
Late_blight: 3113 files
Leaf_Mold: 2754 files
Septoria_leaf_spot: 2882 files
Spider_mites Two-spotted_spider_mite: 1747 files
Target_Spot: 1827 files
Tomato_Yellow_Leaf_Curl_Virus: 2039 files
Tomato_mosaic_virus: 2153 files
healthy: 3051 files
powdery_mildew: 1004 files


In [6]:


def count_files_and_extensions(directory):
    total_files = 0
    extension_counts = {}

    for root, _, files in os.walk(directory):
        for file in files:
            total_files += 1
            ext = os.path.splitext(file)[1].lower()
            extension_counts[ext] = extension_counts.get(ext, 0) + 1

    return total_files, extension_counts

# Assuming base_dataset_path is already defined from previous cells
# base_dataset_path = '/content/drive/MyDrive/plant_disease_dataset/data_/Tomato_'

if 'tomato_dir' in locals() and os.path.exists(tomato_dir):
    total, extensions = count_files_and_extensions(tomato_dir)
    print(f"Total files in dataset: {total}")
    print("File extension breakdown:")
    for ext, count in sorted(extensions.items()):
        print(f"  {ext}: {count}")
else:
    print(f"Base dataset path '{tomato_dir}' not found or not defined.")

Total files in dataset: 32534
File extension breakdown:
  .jpeg: 4
  .jpg: 32025
  .png: 505


#Transfering files from the Drive dir to the Local train dir and validation_dir

In [7]:
import shutil
import os
import tensorflow as tf

# 1. Define paths
drive_train_dir = '/content/drive/MyDrive/plant_disease_dataset/data_/Tomato_/train'
drive_valid_dir = '/content/drive/MyDrive/plant_disease_dataset/data_/Tomato_/valid'

local_train_dir = '/content/local_data/train'
local_valid_dir = '/content/local_data/valid'

# 2. Copy from Drive to Local Runtime
def sync_local_data(src, dst):
    if os.path.exists(src):
        if not os.path.exists(dst):
            print(f"Copying {src} to local runtime...")
            shutil.copytree(src, dst)
        else:
            print(f"Local data already exists at {dst}")

sync_local_data(drive_train_dir, local_train_dir)
sync_local_data(drive_valid_dir, local_valid_dir)

Local data already exists at /content/local_data/train
Local data already exists at /content/local_data/valid


In [8]:
# we are intiating the Dense121 , a highly efficient convolutional neural network
#known for its connecting layer to every other layer

###  include_top=False,
#we chopped out the top layer don't want the model to output standard
#ImageNet classes (like "dog" or "car"); you want raw extracted features so you can add your own custom layers (like CondConv) on top.

 # weights="imagenet",
 #Instead of starting with random weights, the model initializes with weights learned from millions of images on the ImageNet dataset.
base_model=tf.keras.applications.MobileNetV2(
    include_top=False,
    weights="imagenet",
    # Defines the size of the images the model will accept: 224 pixels high, 224 pixels wide, and 3 color channels (Red, Green, Blue).
    input_shape=(224, 224, 3)
)

In [9]:
# --- DATASET LOADING & PREFETCHING ---
batch_size = 16 # Lowered from 32 to 16 to protect GPU VRAM as well
img_height = 224
img_width = 224

print("Loading Training Dataset:")
train_ds = tf.keras.utils.image_dataset_from_directory(
    '/content/local_data/train',
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical',
    shuffle=True
)

print("\nLoading Validation Dataset:")
val_ds = tf.keras.utils.image_dataset_from_directory(
    '/content/local_data/valid',
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical',
    shuffle=False
)

# Optimize memory pipeline WITHOUT .cache() to prevent System RAM crashes
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

Loading Training Dataset:
Found 25851 files belonging to 11 classes.

Loading Validation Dataset:
Found 6683 files belonging to 11 classes.


In [10]:
WEIGHT_DECAY = 2e-4  # A small penalty added to the model's weights to prevent overfitting.


#  his is simply a wrapper function. Instead of typing out all those long arguments (like kernel_initializer="he_normal")
#every time you need a convolutional layer, this function sets up standard defaults for your project.
def conv2d(kernel_size, stride, filters, kernel_regularizer=tf.keras.regularizers.l2(WEIGHT_DECAY), padding="same", use_bias=False,
           kernel_initializer="he_normal", **kwargs):
    return layers.Conv2D(kernel_size=kernel_size, strides=stride, filters=filters, kernel_regularizer=kernel_regularizer, padding=padding,
                         use_bias=use_bias, kernel_initializer=kernel_initializer, **kwargs)

#This layer looks at an incoming image and decides which of the convolution "experts" should be trusted most for this specific image.
class Routing(layers.Layer):
    def __init__(self, out_channels, dropout_rate, temperature=30, **kwargs):
        super(Routing, self).__init__(**kwargs)

        #Squashes the spatial dimensions (height/width) of the
        #image features down to a 1D vector so the network can look at the "global" context of the image.
        self.avgpool = layers.GlobalAveragePooling2D()

        #Randomly turns off neurons during training to prevent the routing mechanism from memorizing the training data.
        self.dropout = layers.Dropout(rate=dropout_rate)

        # A dense layer that outputs a raw score for each "expert" convolution.
        self.fc = layers.Dense(units=out_channels)

        self.softmax = layers.Softmax()

        #A mathematical trick. Dividing by a temperature softens
        #the probability distribution (making the network less "stubborn" about picking just one expert).
        self.temperature = temperature

#The forward pass. It pools the input, applies dropout, calculates the dense scores,
#divides by the temperature, and uses a Softmax function so that the final weights assigned to all
#the experts add up to exactly 1.0 (e.g., Expert 1: 0.6, Expert 2: 0.3, Expert 3: 0.1).
    def call(self, inputs, **kwargs):
        """
        :param inputs: (b, c, h, w)
        :return: (b, out_features)
        """
        out = self.avgpool(inputs)
        out = self.dropout(out)

        # refer to paper: https://arxiv.org/pdf/1912.03458.pdf
        out = self.softmax(self.fc(out) * 1.0 / self.temperature)
        return out


class CondConv2D(layers.Layer):
    def __init__(self, filters, kernel_size, stride=1, use_bias=True, num_experts=3, padding="same", **kwargs):
        super(CondConv2D, self).__init__(**kwargs)

        self.routing = Routing(out_channels=num_experts, dropout_rate=0.2, name="routing_layer")
        self.convs = []
        for _ in range(num_experts):
            self.convs.append(conv2d(filters=filters, stride=stride, kernel_size=kernel_size, use_bias=use_bias, padding=padding))

    def call(self, inputs, **kwargs):
        """
        :param inputs: (b, h, w, c)
        :return: (b, h_out, w_out, filters)
        """
        routing_weights = self.routing(inputs)
        feature = routing_weights[:, 0] * tf.transpose(self.convs[0](inputs), perm=[1, 2, 3, 0])
        for i in range(1, len(self.convs)):
            feature += routing_weights[:, i] * tf.transpose(self.convs[i](inputs), perm=[1, 2, 3, 0])
        feature = tf.transpose(feature, perm=[3, 0, 1, 2])
        return feature

In [11]:
kernel_init = tf.keras.initializers.glorot_uniform()
bias_init = tf.keras.initializers.Constant(value=0.0)

In [12]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.3),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
    # Gaussian noise simulates the graininess of a fast-moving drone camera
    layers.GaussianNoise(0.1)
], name="drone_condition_augmentation")

In [13]:
def Inception(x, nb_filter):
    # Removed the hardcoded name="..." arguments so Keras auto-names them safely
    branch1x1 = CondConv2D(kernel_size=(1,1), filters=nb_filter, stride=1, padding='same', use_bias=True,  num_experts=3)(x)

    branch3x3 = CondConv2D(kernel_size=(1,1), filters=nb_filter, stride=1, padding='same', use_bias=True,  num_experts=3)(x)
    branch3x31 = CondConv2D(kernel_size=(3,1), filters=nb_filter, stride=1, padding='same', use_bias=True,  num_experts=3)(branch3x3)
    branch3x32 = CondConv2D(kernel_size=(1,3), filters=nb_filter, stride=1, padding='same', use_bias=True,  num_experts=3)(branch3x3)
    out1 = layers.Add()([branch3x31, branch3x32])

    branch5x5 = CondConv2D(kernel_size=(1,1), filters=nb_filter, stride=1, padding='same', use_bias=True,  num_experts=3)(x)
    branch5x5_1 = CondConv2D(kernel_size=(3,1), filters=nb_filter, stride=1, padding='same', use_bias=True,  num_experts=3)(branch5x5)
    branch5x5_2 = CondConv2D(kernel_size=(1,3), filters=nb_filter, stride=1, padding='same', use_bias=True,  num_experts=3)(branch5x5)
    out2 = layers.Add()([branch5x5_1, branch5x5_2])

    branch5x51 = CondConv2D(kernel_size=(3,1), filters=nb_filter, stride=1, padding='same', use_bias=True,  num_experts=3)(out2)
    branch5x52 = CondConv2D(kernel_size=(1,3), filters=nb_filter, stride=1, padding='same', use_bias=True,  num_experts=3)(out2)
    out3 = layers.Add()([branch5x51, branch5x52])

    branchpool = MaxPooling2D(pool_size=(3,3), strides=(1,1), padding='same')(x)
    branchpool = CondConv2D(kernel_size=(1,1), filters=nb_filter, stride=1, padding='same', use_bias=True,  num_experts=3)(branchpool)

    # Concatenate all paths together
    x = concatenate([branch1x1, out1, out3, branchpool], axis=3)
    return x

In [14]:
def mlp(x, hidden_units, dropout_rate):
    for units in hidden_units:
        x = layers.Dense(units, activation=tf.nn.gelu)(x)
        x = layers.Dropout(dropout_rate)(x)
    return x

In [15]:
class Patches(layers.Layer):
    def __init__(self, patch_size,**kwargs):
        super(Patches, self).__init__()
        self.patch_size = patch_size

    def get_config(self):
        return {'patch_size': self.patch_size}


    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )
        patch_dims = patches.shape[-1]
        patches = tf.reshape(patches, [batch_size, -1, patch_dims])
        return patches

In [16]:
class PatchEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim,**kwargs):
        super(PatchEncoder, self).__init__()
        self.num_patches = num_patches
        self.projection_dim = projection_dim
        self.projection = layers.Dense(units=projection_dim)
        self.position_embedding = layers.Embedding(input_dim=self.num_patches, output_dim=self.projection_dim)

    def get_config(self):
        return {'num_patches': self.num_patches,
               'projection_dim':self.projection_dim}


    def call(self, patch):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        encoded = self.projection(patch) + self.position_embedding(positions)
        return encoded

In [17]:
def sse_block(input_feature, ratio=4):
    """Implementation of Squeeze-and-Excitation(SE) block using Keras Layers."""
    channel_axis = -1
    channel = input_feature.shape[channel_axis]

    # Squeeze & Excitation Path
    se_feature = layers.GlobalAveragePooling2D()(input_feature)
    se_feature = layers.Reshape((1, 1, channel))(se_feature)
    se_feature = layers.Dense(channel // ratio, activation='relu', kernel_initializer='he_normal')(se_feature)
    se_feature = layers.Dense(channel, activation='sigmoid', kernel_initializer='he_normal')(se_feature)

    # Multiply the input by the SE features
    se_out = layers.Multiply()([input_feature, se_feature])

    # Spatial Statistics (Mean, Std, Max)
    # We wrap tf functions in Lambda layers to avoid the KerasTensor error
    mean = layers.Lambda(lambda x: tf.reduce_mean(x, axis=-1, keepdims=True))(input_feature)
    std = layers.Lambda(lambda x: tf.math.reduce_std(x, axis=-1, keepdims=True))(input_feature)
    maximum = layers.Lambda(lambda x: tf.reduce_max(x, axis=-1, keepdims=True))(input_feature)

    # Final Concatenation
    out = layers.Concatenate()([se_out, mean, std, maximum])

    return out

In [18]:
image_size =56
patch_size = 5  # Size of the patches to be extract from the input images
num_patches = (image_size // patch_size) ** 2
projection_dim = 32
num_heads = 4
transformer_units = [
    projection_dim * 2,
    projection_dim,
]  # Size of the transformer layers
transformer_layers = 4


In [19]:
# --- 1. SET UP HYPERPARAMETERS ---
image_size = 56
patch_size = 5
num_patches = (image_size // patch_size) ** 2
projection_dim = 32
num_heads = 4
transformer_units = [projection_dim * 2, projection_dim]
transformer_layers = 4
TOTAL_CLASSES = 11

# --- 2. AUGMENTED GRAPH TRICK ---
inputs = tf.keras.Input(shape=(224, 224, 3))
augmented_x = data_augmentation(inputs)

base_model = tf.keras.applications.MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_tensor=augmented_x
)

# --- 3. MULTI-SCALE FEATURE EXTRACTION ---
# Branch 1: High-level features. Output from base is 112x112.
x1 = base_model.get_layer('expanded_conv_project_BN').output
# Use stride 2 and 'same' padding to reach target size 11x11 incrementally
x1 = CondConv2D(kernel_size=3, filters=16, stride=2, padding='same', num_experts=3)(x1) # 56x56
x1 = CondConv2D(kernel_size=3, filters=32, stride=2, padding='same', num_experts=3)(x1) # 28x28
x1 = CondConv2D(kernel_size=3, filters=32, stride=2, padding='same', num_experts=3)(x1) # 14x14
# FIX: filters=29 so that sse_block outputs exactly 32 channels
x1 = Conv2D(kernel_size=4, filters=29, strides=1, padding='valid')(x1)
x_conv1 = sse_block(x1, ratio=4)

# Branch 2: Mid-level features. Output from base is 56x56.
x2 = base_model.get_layer('block_2_project_BN').output
x2 = CondConv2D(kernel_size=3, filters=16, stride=2, padding='same', num_experts=3)(x2) # 28x28
x2 = CondConv2D(kernel_size=3, filters=32, stride=2, padding='same', num_experts=3)(x2) # 14x14
# FIX: filters=29
x2 = Conv2D(kernel_size=4, filters=29, strides=1, padding='valid')(x2)
x_conv2 = sse_block(x2, ratio=4)

# Branch 3: Deep features + Inception. Output from base is 28x28.
x_inc = base_model.get_layer('block_5_add').output
x_inc_out = Inception(x_inc, 32)
x3 = CondConv2D(kernel_size=3, filters=32, stride=2, padding='same', num_experts=3)(x_inc_out) # 14x14
# FIX: filters=29
x3 = Conv2D(kernel_size=4, filters=29, strides=1, padding='valid')(x3)
x_conv3 = sse_block(x3, ratio=4)

# --- 4. VISION TRANSFORMER PATH ---
patches = Patches(patch_size)(x_inc_out)
encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)

for _ in range(transformer_layers):
    x_norm1 = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
    attention_output = layers.MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim, dropout=0.1)(x_norm1, x_norm1)
    x_add1 = layers.Add()([attention_output, encoded_patches])
    x_norm2 = layers.LayerNormalization(epsilon=1e-6)(x_add1)
    x_mlp = mlp(x_norm2, hidden_units=transformer_units, dropout_rate=0.1)
    encoded_patches = layers.Add()([x_mlp, x_add1])

x_final_trans = layers.LayerNormalization(epsilon=1e-6, name='cam_layer')(encoded_patches)
x_t = layers.Reshape((11, 11, 32))(x_final_trans)

# --- 5. FUSION & CLASSIFICATION ---
# Now all inputs to Add() are exactly (11, 11, 32)
merged = layers.Add()([x_conv1, x_conv2, x_conv3, x_t])
merged = sse_block(merged, ratio=4)
merged = layers.GlobalAveragePooling2D()(merged)
predictions = Dense(TOTAL_CLASSES, activation='softmax')(merged)

drone_model = Model(inputs=inputs, outputs=predictions)
drone_model.summary()

/tmp/ipykernel_36367/3121104240.py:15: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drone_condition_au… │ (None, 224, 224,  │          0 │ input_layer_1[0]… │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ drone_condition_… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis

 Total params: 550,657 (2.10 MB)

 Trainable params: 547,137 (2.09 MB)

 Non-trainable params: 3,520 (13.75 KB)

In [ ]:
import os
import tensorflow as tf
from tqdm.keras import TqdmCallback # Import the tqdm keras integration

# 1. Define checkpoint path directly inside your Google Drive folder
checkpoint_dir = '/content/drive/MyDrive/plant_disease_dataset/checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_path = os.path.join(checkpoint_dir, 'best_drone_model.keras')

# 2. Compile Model
drone_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.CategoricalFocalCrossentropy(alpha=0.25, gamma=2.0),
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 3. Auto-Resume Check
if os.path.exists(checkpoint_path):
    print(f"Found existing checkpoint at '{checkpoint_path}'. Loading model weights to resume...")
    try:
        drone_model.load_weights(checkpoint_path)
        print("Weights successfully restored! Ready to resume training.")
    except Exception as e:
        print(f" Could not load weights directly ({e}). Starting fresh baseline training.")
else:
    print("No previous checkpoint found on Google Drive. Starting fresh baseline training...")

# 4. Define Callbacks
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    monitor='val_loss',
    save_best_only=True,
    verbose=0 # Turn off keras print so it doesn't fight with tqdm
)

early_stopping_cb = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True
)

# Initialize the beautiful TQDM progress bar
tqdm_callback = TqdmCallback(verbose=1)

# 5. Start / Resume Training Loop
epochs = 30
history = drone_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs,
    verbose=0, # TURN OFF the default Keras progress bar
    callbacks=[checkpoint_cb, early_stopping_cb, tqdm_callback]
)

No previous checkpoint found on Google Drive. Starting fresh baseline training...


0epoch [00:00, ?epoch/s]

0batch [00:00, ?batch/s]